# Week 15 — End-to-End Machine Learning (Classroom Notebook)

> 跟著 Géron 章節 2 的 California Housing 案例，把 ML pipeline 走一遍。
> 預期執行時間：~45 分鐘（含等 model 訓練）。

**Setup**：本 notebook 假設你的環境已裝好 `scikit-learn>=1.4`、`pandas`、`numpy`、`matplotlib`。


In [1]:
# 環境檢查
import sklearn, pandas as pd, numpy as np, matplotlib
print('sklearn:', sklearn.__version__)
print('pandas :', pd.__version__)
print('numpy  :', np.__version__)


sklearn: 1.8.0
pandas : 2.3.3
numpy  : 2.4.2


## 1. Load the data
California Housing 是 1990 美國加州人口普查的 block-level 資料。我們的目標是預測 `median_house_value`。

In [2]:
import tarfile, urllib.request
from pathlib import Path

def load_housing_data():
    tarball_path = Path('datasets/housing.tgz')
    if not tarball_path.is_file():
        Path('datasets').mkdir(parents=True, exist_ok=True)
        url = 'https://github.com/ageron/data/raw/main/housing.tgz'
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as t:
            t.extractall(path='datasets')
    return pd.read_csv(Path('datasets/housing/housing.csv'))

housing = load_housing_data()
print(housing.shape)
housing.head()

(20640, 10)


C:\Users\audachang\AppData\Local\Temp\ipykernel_27244\2153167283.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(path='datasets')


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


**注意**：`total_bedrooms` 只有 20433 個非空值（vs. 其他欄位 20640）— 之後需要 impute。

In [4]:
housing.describe().round(2)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.00,20640.00,20640.00,20640.00,20433.00,20640.00,20640.00,20640.00,20640.00
mean,-119.57,35.63,28.64,2635.76,537.87,1425.48,499.54,3.87,206855.82
std,2.00,2.14,12.59,2181.62,421.39,1132.46,382.33,1.90,115395.62
min,-124.35,32.54,1.00,2.00,1.00,3.00,1.00,0.50,14999.00
25%,-121.80,33.93,18.00,1447.75,296.00,787.00,280.00,2.56,119600.00
50%,-118.49,34.26,29.00,2127.00,435.00,1166.00,409.00,3.53,179700.00
75%,-118.01,37.71,37.00,3148.00,647.00,1725.00,605.00,4.74,264725.00
max,-114.31,41.95,52.00,39320.00,6445.00,35682.00,6082.00,15.00,500001.00


## 2. 切 test set（**第一件事**）

如果我們先做 EDA 再切 test，可能會被視覺暗示影響到設計選擇 → data snooping bias。


In [5]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)
print(f'Train: {len(train_set)}, Test: {len(test_set)}')

Train: 16512, Test: 4128


### Stratified split — 修正 income 分布偏差

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

housing['income_cat'] = pd.cut(
    housing['median_income'],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5],
)

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
for tr_idx, te_idx in split.split(housing, housing['income_cat']):
    strat_train = housing.iloc[tr_idx].drop('income_cat', axis=1)
    strat_test  = housing.iloc[te_idx].drop('income_cat', axis=1)

# 比較分層 vs. 隨機 vs. 母體的 income_cat 比例
prop = pd.DataFrame({
    'overall': housing['income_cat'].value_counts(normalize=True).sort_index(),
    'strat'  : pd.cut(strat_test['median_income'],
                      bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                      labels=[1,2,3,4,5]).value_counts(normalize=True).sort_index(),
})
prop['diff_strat'] = (prop['strat'] - prop['overall']).abs()
prop.round(3)

,overall,strat,diff_strat
1,0.040,0.040,0.0
2,0.319,0.319,0.0
3,0.351,0.351,0.0
4,0.176,0.176,0.0
5,0.114,0.114,0.0


### 🔬 Hands-on 1
試試看：如果用 `random_state=0`（而非 42）做 stratified split，前述比例會變嗎？

<details><summary>💡 提示</summary>不會 — stratification 確保比例與母體一致，與 seed 無關（變的只是哪些 sample 被選）。</details>

## 3. EDA — 在 train set 上做

In [ ]:
import matplotlib.pyplot as plt
strat_train.hist(bins=50, figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
# 與 target 的 correlation
corr = strat_train.corr(numeric_only=True)['median_house_value'].sort_values(ascending=False)
print(corr)

In [ ]:
# Geographic scatter — 點大小=人口，顏色=房價
strat_train.plot(kind='scatter', x='longitude', y='latitude',
                 alpha=0.3, s=strat_train['population']/100,
                 c='median_house_value', cmap='jet', figsize=(10, 6),
                 colorbar=True, label='population')
plt.show()

## 4. Pipeline — preprocessing without leakage

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_attribs = ['longitude', 'latitude', 'housing_median_age',
               'total_rooms', 'total_bedrooms', 'population',
               'households', 'median_income']
cat_attribs = ['ocean_proximity']

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

prep = ColumnTransformer([
    ('num', num_pipe, num_attribs),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_attribs),
])

X_train = strat_train.drop('median_house_value', axis=1)
y_train = strat_train['median_house_value']
X_train_prepared = prep.fit_transform(X_train)
print('Prepared shape:', X_train_prepared.shape)

### 🔬 Hands-on 2
**任務**：把 `ocean_proximity` 改用 `OrdinalEncoder` 而非 `OneHotEncoder`，看 RandomForest CV RMSE 是否會變？

<details><summary>💡 提示</summary>
類別本身無自然順序，OrdinalEncoder 會把 `<1H OCEAN`、`INLAND`、`ISLAND` 等強加數字順序 — 對 linear model 是災難，但 tree-based 對此較不敏感。
</details>

## 5. Algorithm zoo — 五大類 regression model 對決

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

models = {
    'Linear'       : LinearRegression(),
    'Ridge'        : Ridge(alpha=1.0),
    'k-NN(5)'      : KNeighborsRegressor(n_neighbors=5),
    'DecisionTree' : DecisionTreeRegressor(random_state=42),
    'RandomForest' : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradBoost'    : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

# 注意：SVR 在 16k samples 上跑太慢，這裡跳過（或自己 subsample）
results = {}
for name, model in models.items():
    pipe = make_pipeline(prep, model)
    rmse = -cross_val_score(pipe, X_train, y_train, cv=5,
                            scoring='neg_root_mean_squared_error', n_jobs=-1)
    results[name] = rmse
    print(f'{name:<14s} RMSE = {rmse.mean():>7.0f} ± {rmse.std():>5.0f}')

In [ ]:
# 視覺化
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
means = [results[n].mean() for n in names]
stds  = [results[n].std()  for n in names]
ax.bar(names, means, yerr=stds, capsize=4)
ax.set_ylabel('5-fold CV RMSE (USD)')
ax.set_title('Algorithm comparison on California Housing')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 6. Hyperparameter tuning — `RandomizedSearchCV`

In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

pipe = make_pipeline(prep, RandomForestRegressor(random_state=42, n_jobs=-1))
param_dist = {
    'randomforestregressor__n_estimators': randint(50, 300),
    'randomforestregressor__max_features': randint(2, 9),
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=8, cv=3,
                            scoring='neg_root_mean_squared_error',
                            random_state=42, n_jobs=-1)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print(f'Best CV RMSE: {-search.best_score_:.0f}')

## 7. Final test — only ONCE

In [ ]:
from sklearn.metrics import root_mean_squared_error
final = search.best_estimator_
X_test = strat_test.drop('median_house_value', axis=1)
y_test = strat_test['median_house_value']
test_rmse = root_mean_squared_error(y_test, final.predict(X_test))
print(f'Final TEST RMSE: {test_rmse:.0f}')

## 8. Unsupervised taster — IsolationForest for outliers

In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=42)
flags = iso.fit_predict(strat_train[num_attribs].dropna())
print(f'Flagged {(flags == -1).sum()} / {len(flags)} as outliers '
      f'({(flags == -1).mean():.1%}).')

## 9. Cogneuro mini-example — Stroop RT
跑一遍合成的 trial-level RT 資料，看同樣的 pipeline 也能用。

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)
def simulate_stroop(n_subj=200, n_trial=30, interaction=False):
    rows = []
    for sid in range(n_subj):
        age = rng.uniform(20, 75)
        for t in range(n_trial):
            congruent = rng.random() < 0.5
            isi = rng.choice([400, 800, 1200])
            cong_effect = 0 if congruent else 60
            if interaction and not congruent:
                cong_effect += 1.5 * (age - 45)
            rt = 350 + 2.0*(age-45) + cong_effect - 0.02*isi + rng.normal(0, 40)
            rows.append((sid, age, congruent, isi, t, rt))
    return pd.DataFrame(rows, columns=['sid','age','congruent','isi','trial_num','rt'])

df = simulate_stroop(interaction=True)
df.head()

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
df['age_bin'] = pd.cut(df['age'], bins=[20,35,50,65,75], labels=[1,2,3,4])
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for tr, te in sss.split(df, df['age_bin']):
    train_rt, test_rt = df.iloc[tr], df.iloc[te]

rt_num = ['age', 'isi', 'trial_num']
rt_cat = ['congruent']
rt_prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc',  StandardScaler())]), rt_num),
    ('cat', OneHotEncoder(), rt_cat),
])

X_tr = train_rt[rt_num + rt_cat]; y_tr = train_rt['rt']
for name, model in [('Linear', LinearRegression()),
                    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))]:
    pipe = make_pipeline(rt_prep, model)
    rmse = -cross_val_score(pipe, X_tr, y_tr, cv=5,
                            scoring='neg_root_mean_squared_error', n_jobs=-1)
    print(f'{name:<14s} RMSE = {rmse.mean():.1f} ± {rmse.std():.1f} ms')

### 🔬 Hands-on 3 — 最後挑戰
把 `simulate_stroop(interaction=False)` 重跑一遍。這次 linear vs. RandomForest 結果如何變化？解釋為什麼。

<details><summary>✅ 預期答案</summary>
`interaction=False` 時資料是線性的，`LinearRegression` 應該與 RandomForest 接近甚至略勝（RF 多餘的容量會增加 variance）。這示範了 **「最佳 model 取決於資料的真實結構」**。
</details>

---

## Take-aways

1. **永遠先切 test set**，並且 **只評估一次**。
2. **`Pipeline` 是 leakage 防火牆** — fit on train, transform on test。
3. **沒有單一最強 algorithm**：linear / tree / ensemble / k-NN / kernel 各有所長，要靠 CV 比較。
4. **Ensemble (RandomForest, GradientBoosting) 是 tabular data 的強力 baseline**。
5. **Unsupervised methods (k-means, IsolationForest) 可作為 feature engineering 或 outlier filter**。
